This is the notebook to define the 8 Qubit QAOA in Bloqade SDK using python:

We need Hc, which is the 8 values that come out of H function after passing it the feateures variables

We need Hm, which we define it, a common way to define it is using RX gate (rotations in X)

We need to define the QAOA algorithm, that in essence is: 
- Starting the 8 qubits
- Apply Hadamard gates to create equal superposition
- Apply Hc part of the circuit which would be quantum gates 
- Apply Hm part of the circuit which would be quantum gates
- Measure it   

** After measurement you get a bitstring **

After having your QAOA defined, we also have to define auxiliar functions in python to make x quantity of shots per circuit, define the probability of a bitstring, the function to calculate energy/cost of a bitstring, a classical optimizer to update parameters from QAOA, run the circuit again, so we need the function to run the circuit

**pip installs [ I DO NOT USE THESE, USE A PYTHON ENVIRONMENT USING POWERSHELL, CHECK THE CELL BELOW]**

In [ ]:
# pip install -q bloqade 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import sys
# !{sys.executable} -m pip install -U lark

In [ ]:
# from bloqade import tsim
# print("tsim ok")

**INSTALL using powershell dependencies**

## Environment setup (PowerShell, one-time)

This notebook is intended to run in the `bloqade312` environment on Windows using VS Code `.ipynb`.

### 1) Create and activate the environment
Run these commands in **PowerShell**:

```powershell
py install 3.12
py -3.12 -m venv C:\bloqade312
C:\bloqade312\Scripts\Activate.ps1

## 2) Install required packages

```powershell
python -m pip install --upgrade pip
pip install bloqade ipykernel scipy numpy
python -m ipykernel install --user --name bloqade312 --display-name "Python (bloqade312)"
```

## 3) In VS Code

Open the notebook and select the kernel:

`Python (bloqade312)`

In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\bloqade312\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [2]:
import math
from typing import Any
import numpy as np
import kirin
from kirin.dialects import ilist
from bloqade import qasm2
from bloqade import squin, tsim
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList

In [3]:
from bloqade import squin
print("squin ok")

squin ok


In [4]:
from bloqade import tsim
print("tsim ok")

tsim ok


In [5]:
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
from typing import Any

Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

print("types ok")

types ok


In [ ]:
"""from bloqade import squin
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
from typing import Any

Measurement = IList[MeasurementResult, Any]

@squin.kernel
def bell() -> Measurement:
    q = squin.qalloc(2)
    squin.h(q[0])
    squin.cx(q[0], q[1])
    bits = squin.broadcast.measure(q)
    return bits

print("kernel created")"""

**Imports**

In [6]:
import math
from typing import Any
import numpy as np

from bloqade import qasm2
from bloqade import squin, tsim
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList

pi = math.pi

Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

print("squin/tsim imports worked")

squin/tsim imports worked


**Global variables from Hamiltonian function we produced**

With these variables produced already, we do not need to use the dataset again or touch it in this ipynb because we already produced the hamiltonian cost on the variables we are using.

The ground truth was produced by brute forcing the 256 combinations and checking the best one, since is easy for a classical computer to do that 

In [8]:
n_qubits = 8

qubit_labels = [
    "A004_LSB", "A004_MSB",
    "A047_LSB", "A047_MSB",
    "A026_LSB", "A026_MSB",
    "A017_LSB", "A017_MSB",
]

# Constant offset is irrelevant for QAOA optimization because it does not change
# which bitstring minimizes the objective.
c0 = 0.0

# FIXED H terms
# Local fields h_i for H_C = sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
h_terms = [
    (0, -0.019730),
    (1, -0.039461),
    (2, -0.014721),
    (3, -0.029441),
    (4, -0.028659),
    (5, -0.057318),
    (6, -0.047267),
    (7, -0.094534),
]

# FIXED J terms
J_terms = [
    (0, 1, 0.007132),
    (0, 2, 0.002667),
    (0, 3, 0.005334),
    (0, 4, 0.005334),
    (0, 5, 0.010667),
    (0, 6, 0.008889),
    (0, 7, 0.017778),
    (1, 2, 0.005334),
    (1, 3, 0.010667),
    (1, 4, 0.010667),
    (1, 5, 0.021335),
    (1, 6, 0.017778),
    (1, 7, 0.035555),
    (2, 3, 0.004005),
    (2, 4, 0.004000),
    (2, 5, 0.008000),
    (2, 6, 0.006667),
    (2, 7, 0.013335),
    (3, 4, 0.008000),
    (3, 5, 0.016000),
    (3, 6, 0.013335),
    (3, 7, 0.026665),
    (4, 5, 0.016005),
    (4, 6, 0.013335),
    (4, 7, 0.026667),
    (5, 6, 0.026667),
    (5, 7, 0.053335),
    (6, 7, 0.044450),
]


# Remains the same even after fix
ground_truth_bitstring = "11011110"


**First we are going to convert the Hamiltonian data to arrays**

In [9]:
import numpy as np

# Build dense h vector and J matrix from the term lists
h8 = np.zeros(n_qubits, dtype=float)
J8 = np.zeros((n_qubits, n_qubits), dtype=float)

for i, hi in h_terms:
    h8[i] = hi

for i, j, Jij in J_terms:
    J8[i, j] = Jij
    J8[j, i] = Jij

ising_offset = 0.0

print("h8:")
print(h8)
print("\nJ8:")
print(J8)

h8:
[-0.01973  -0.039461 -0.014721 -0.029441 -0.028659 -0.057318 -0.047267
 -0.094534]

J8:
[[0.       0.007132 0.002667 0.005334 0.005334 0.010667 0.008889 0.017778]
 [0.007132 0.       0.005334 0.010667 0.010667 0.021335 0.017778 0.035555]
 [0.002667 0.005334 0.       0.004005 0.004    0.008    0.006667 0.013335]
 [0.005334 0.010667 0.004005 0.       0.008    0.016    0.013335 0.026665]
 [0.005334 0.010667 0.004    0.008    0.       0.016005 0.013335 0.026667]
 [0.010667 0.021335 0.008    0.016    0.016005 0.       0.026667 0.053335]
 [0.008889 0.017778 0.006667 0.013335 0.013335 0.026667 0.       0.04445 ]
 [0.017778 0.035555 0.013335 0.026665 0.026667 0.053335 0.04445  0.      ]]


**Classical Icing Energy**

In [10]:
def ising_energy_from_x_list(x_list, h, J, ising_offset=0.0):
    # Use the convention x = (1 + s)/2, so s = 2x - 1
    s = np.array([2 * int(bit) - 1 for bit in x_list], dtype=float)

    energy = ising_offset

    for i in range(len(h)):
        energy += h[i] * s[i]

    for i in range(J.shape[0]):
        for j in range(i + 1, J.shape[1]):
            energy += J[i, j] * s[i] * s[j]

    return float(energy)

**Helper to convert bitstring to x-list**

In [11]:
def bitstring_to_x_list(bitstring):
    return [int(b) for b in bitstring]

**Decode 2 bits per asset**

In [12]:
def decode_lots_from_bitstring(bitstring):
    """
    Decode 8 qubits into 4 asset lot values using:
    lot = LSB + 2 * MSB
    """
    bits = [int(b) for b in bitstring]
    lots = []

    for k in range(0, len(bits), 2):
        lsb = bits[k]
        msb = bits[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)

    return lots

**Verify the portfolio**

In [13]:
decoded_lots = decode_lots_from_bitstring(ground_truth_bitstring)
print("Decoded lots from ground-truth bitstring:", decoded_lots)

Decoded lots from ground-truth bitstring: [3, 2, 3, 1]


**Test the ground truth**

In [14]:
ground_truth_x = [1, 1, 0, 1, 1, 1, 1, 0]

print("Ground-truth x:", ground_truth_x)
print("Ground-truth lots:", decode_lots_from_bitstring("".join(str(b) for b in ground_truth_x)))
print("Ground-truth corrected Ising energy:", ising_energy_from_x_list(ground_truth_x, h8, J8, ising_offset))

Ground-truth x: [1, 1, 0, 1, 1, 1, 1, 0]
Ground-truth lots: [3, 2, 3, 1]
Ground-truth corrected Ising energy: -0.14326400000000006


**Brute-force classical check over all 256 bitstrings**

In [15]:
def x_list_to_lots(x_list):
    lots = []
    for k in range(0, len(x_list), 2):
        lsb = x_list[k]
        msb = x_list[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)
    return lots

all_results_rebuilt = []

for num in range(2 ** n_qubits):
    x_list = [(num >> i) & 1 for i in range(n_qubits)]
    e_ising = ising_energy_from_x_list(x_list, h8, J8, ising_offset)

    all_results_rebuilt.append({
        "x": x_list,
        "lots": x_list_to_lots(x_list),
        "ising_energy": e_ising
    })

all_results_rebuilt = sorted(all_results_rebuilt, key=lambda r: r["ising_energy"])

print("Top 15 solutions ranked by corrected Ising energy:\n")
for rank, r in enumerate(all_results_rebuilt[:15], start=1):
    print(f"{rank:2d}. x={r['x']}  lots={r['lots']}  Ising={r['ising_energy']:.8f}")

Top 15 solutions ranked by corrected Ising energy:

 1. x=[1, 1, 0, 1, 1, 1, 1, 0]  lots=[3, 2, 3, 1]  Ising=-0.14326400
 2. x=[0, 1, 1, 1, 1, 1, 1, 0]  lots=[2, 3, 3, 1]  Ising=-0.14306000
 3. x=[1, 1, 1, 1, 0, 1, 1, 0]  lots=[3, 3, 2, 1]  Ising=-0.14206000
 4. x=[1, 1, 1, 1, 1, 0, 0, 1]  lots=[3, 3, 1, 2]  Ising=-0.14195000
 5. x=[1, 1, 0, 1, 1, 0, 0, 1]  lots=[3, 2, 1, 2]  Ising=-0.14185600
 6. x=[0, 1, 0, 1, 0, 1, 0, 1]  lots=[2, 2, 2, 2]  Ising=-0.14108200
 7. x=[1, 0, 1, 1, 0, 1, 0, 1]  lots=[1, 3, 2, 2]  Ising=-0.14082800
 8. x=[1, 1, 1, 0, 0, 0, 1, 1]  lots=[3, 1, 0, 3]  Ising=-0.14071800
 9. x=[1, 1, 1, 0, 1, 1, 1, 0]  lots=[3, 1, 3, 1]  Ising=-0.14050000
10. x=[1, 1, 1, 0, 0, 1, 0, 1]  lots=[3, 1, 2, 2]  Ising=-0.14038400
11. x=[1, 1, 0, 0, 0, 1, 0, 1]  lots=[3, 0, 2, 2]  Ising=-0.14027000
12. x=[0, 1, 1, 1, 1, 0, 0, 1]  lots=[2, 3, 1, 2]  Ising=-0.13986800
13. x=[0, 1, 1, 1, 0, 0, 1, 1]  lots=[2, 3, 0, 3]  Ising=-0.13972600
14. x=[0, 1, 0, 1, 0, 0, 1, 1]  lots=[2, 2, 0, 3]  

**Check the rank of the known ground truth**

In [16]:
gt_rank = None

for idx, r in enumerate(all_results_rebuilt):
    if r["x"] == ground_truth_x:
        gt_rank = idx + 1
        print("Ground-truth entry found:")
        print(r)
        break

print("\nGround-truth rank under corrected Ising evaluation:", gt_rank)

Ground-truth entry found:
{'x': [1, 1, 0, 1, 1, 1, 1, 0], 'lots': [3, 2, 3, 1], 'ising_energy': -0.14326400000000006}

Ground-truth rank under corrected Ising evaluation: 1


# We actually create QAOA now as a bloqade quantum circuit

Import the simulator


In [17]:
from bloqade.pyqrack import StackMemorySimulator

**Build the QAOA algorithm on Bloqade**


In [18]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        # Initialize the register in the |+> state
        for i in range(n_qubits):
            qasm2.h(q[i])

        # Apply p QAOA layers
        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            # Single-qubit Z terms
            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            # Two-qubit ZZ terms
            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            # Mixer layer
            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

**Create the Kernel**

In [19]:
kernel = build_qaoa_ising_kernel(n_qubits, h_terms, J_terms)
print(kernel)

Method("kernel")


In [20]:
print(kernel)
print(type(kernel))

Method("kernel")
<class 'kirin.ir.method.Method'>


**Choose initial QAOA parameters**

In [21]:
# P stands for the depth of the circuit, how many times we repeat the gates
p = 1

gamma_init = [0.3] * p
beta_init = [0.2] * p

**wrap in main and emit QASM**

In [22]:
@qasm2.extended
def main():
    return kernel(gamma_init, beta_init)

**print the generated circuit**

In [23]:
target = qasm2.emit.QASM2()
ast = target.emit(main)
qasm2.parse.pprint(ast)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[8];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
rz (-0.011838) q[0];
rz (-0.023676600000000003) q[1];
rz (-0.0088326) q[2];
rz (-0.0176646) q[3];
rz (-0.0171954) q[4];
rz (-0.0343908) q[5];
rz (-0.028360200000000002) q[6];
rz (-0.056720400000000004) q[7];
CX q[0], q[1];
rz (0.0042792) q[1];
CX q[0], q[1];
CX q[0], q[2];
rz (0.0016002) q[2];
CX q[0], q[2];
CX q[0], q[3];
rz (0.0032004) q[3];
CX q[0], q[3];
CX q[0], q[4];
rz (0.0032004) q[4];
CX q[0], q[4];
CX q[0], q[5];
rz (0.006400199999999999) q[5];
CX q[0], q[5];
CX q[0], q[6];
rz (0.005333399999999999) q[6];
CX q[0], q[6];
CX q[0], q[7];
rz (0.010666799999999999) q[7];
CX q[0], q[7];
CX q[1], q[2];
rz (0.0032004) q[2];
CX q[1], q[2];
CX q[1], q[3];
rz (0.006400199999999999) q[3];
CX q[1], q[3];
CX q[1], q[4];
rz (0.006400199999999999) q[4];
CX q[1], q[4];
CX q[1], q[5];
rz (0.012801) q[5];
CX q[1], q[5];
CX q[1], q[6];
rz (0.010666799999999999) q[6];
CX q[1], q[6];
CX q

**run the circuit and get the state vector**

In [24]:
sim = StackMemorySimulator(min_qubits=n_qubits)

ket = sim.state_vector(main)

print("State vector length:", len(ket))
print("Number of qubits:", n_qubits)

State vector length: 256
Number of qubits: 8


**Convert the state vector into probabilities**

In [25]:
probs = np.abs(np.array(ket))**2

print("Probability sum:", probs.sum())

Probability sum: 1.0000000064000112


**helper to show the most likely bitstrings**

In [26]:
def index_to_bitstring(idx, n):
    return format(idx, f"0{n}b")

top_k = 15
top_indices = np.argsort(probs)[::-1][:top_k]

print(f"Top {top_k} most likely computational basis states:\n")
for rank, idx in enumerate(top_indices, start=1):
    bitstring = index_to_bitstring(idx, n_qubits)
    prob = probs[idx]
    print(f"{rank:2d}. bitstring={bitstring}   prob={prob:.8f}")

Top 15 most likely computational basis states:

 1. bitstring=11111111   prob=0.00503798
 2. bitstring=11111011   prob=0.00482711
 3. bitstring=11111110   prob=0.00476186
 4. bitstring=11101111   prob=0.00464129
 5. bitstring=11110111   prob=0.00463960
 6. bitstring=11111010   prob=0.00458252
 7. bitstring=11111101   prob=0.00452577
 8. bitstring=11101011   prob=0.00447629
 9. bitstring=11110011   prob=0.00447469
10. bitstring=11101110   prob=0.00442547
11. bitstring=10111111   prob=0.00442532
12. bitstring=11110110   prob=0.00442385
13. bitstring=11111001   prob=0.00437450
14. bitstring=00000000   prob=0.00435507
15. bitstring=11100111   prob=0.00433081


**check the ground-truth probability**

In [27]:
ground_truth_bitstring = "11011110"
gt_index = int(ground_truth_bitstring, 2)

print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth probability:", probs[gt_index])

Ground-truth bitstring: 11011110
Ground-truth probability: 0.004163693891188598


**Compute ising energy for the most likely bitstrings**

In [28]:
def bitstring_to_x_list(bitstring):
    return [int(b) for b in bitstring]

print("Top states with Ising energy:\n")
for rank, idx in enumerate(top_indices, start=1):
    bitstring = index_to_bitstring(idx, n_qubits)
    x_list = bitstring_to_x_list(bitstring)
    energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
    print(f"{rank:2d}. bitstring={bitstring}   prob={probs[idx]:.8f}   energy={energy:.8f}")

Top states with Ising energy:

 1. bitstring=11111111   prob=0.00503798   energy=0.10847200
 2. bitstring=11111011   prob=0.00482711   energy=-0.08091000
 3. bitstring=11111110   prob=0.00476186   energy=-0.13803000
 4. bitstring=11101111   prob=0.00464129   energy=-0.00065800
 5. bitstring=11110111   prob=0.00463960   energy=-0.00222600
 6. bitstring=11111010   prob=0.00458252   energy=-0.11407200
 7. bitstring=11111101   prob=0.00452577   energy=-0.05923600
 8. bitstring=11101011   prob=0.00447629   energy=-0.12604000
 9. bitstring=11110011   prob=0.00447469   energy=-0.12758800
10. bitstring=11101110   prob=0.00442547   energy=-0.14050000
11. bitstring=10111111   prob=0.00442532   energy=-0.02954200
12. bitstring=11110110   prob=0.00442385   energy=-0.14206000
13. bitstring=11111001   prob=0.00437450   energy=-0.14195000
14. bitstring=00000000   prob=0.00435507   energy=0.77073400
15. bitstring=11100111   prob=0.00433081   energy=-0.07935600


NO LE ENTIENDO ES CHAT: Cell 8 — compare direct and reversed bitstring conventions

In [29]:
ground_truth_x = [1, 1, 0, 1, 1, 1, 1, 0]

gt_direct = "".join(str(b) for b in ground_truth_x)
gt_reversed = gt_direct[::-1]

print("Direct ground-truth bitstring:  ", gt_direct, "prob =", probs[int(gt_direct, 2)])
print("Reversed ground-truth bitstring:", gt_reversed, "prob =", probs[int(gt_reversed, 2)])

Direct ground-truth bitstring:   11011110 prob = 0.004163693891188598
Reversed ground-truth bitstring: 01111011 prob = 0.003938217110797988


**Everything works natural and good now, the circuit is working, now we start doing the classical optimization of the 2 variatonal parameters of the circuit, gamma and beta, so we can start getting good results**

Right now i have hard-coded gamma and beta 

In [30]:
def build_main_from_params(gamma_vals, beta_vals):
    @qasm2.extended
    def main():
        return kernel(gamma_vals, beta_vals)
    return main

In [31]:
def expected_ising_energy(gamma_vals, beta_vals):
    main = build_main_from_params(gamma_vals, beta_vals)
    ket = sim.state_vector(main)
    probs = np.abs(np.array(ket)) ** 2

    exp_energy = 0.0
    for idx in range(2 ** n_qubits):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
        exp_energy += probs[idx] * energy

    return float(exp_energy)

In [39]:
print("Expected energy at gamma=0.3, beta=0.2:",
      expected_ising_energy([0.3], [0.2]))

Expected energy at gamma=0.3, beta=0.2: -0.0005120791314177489


In [40]:
from scipy.optimize import minimize
eval_history = []

def objective_theta(theta):
    gamma_vals = [theta[0]]
    beta_vals  = [theta[1]]

    value = expected_ising_energy(gamma_vals, beta_vals)
    eval_history.append((theta[0], theta[1], value))

    print(f"gamma={theta[0]:.6f}, beta={theta[1]:.6f}, expected_energy={value:.8f}")
    return value

**CLASSICAL OPTIMIZER to update params on variational quantum circuit**

In [46]:
theta0 = np.array([0.3, 0.2], dtype=float)

result = minimize(
    objective_theta,
    theta0,
    method="COBYLA",
    options={"maxiter": 1000, "rhobeg": 0.2}
)

print("\nOptimization finished")
print("Best theta:", result.x)
print("Best expected energy:", result.fun)
print("Success:", result.success)
print("Message:", result.message)

gamma=0.300000, beta=0.200000, expected_energy=-0.00051210
gamma=0.500000, beta=0.200000, expected_energy=-0.00078981
gamma=0.500000, beta=0.400000, expected_energy=-0.00266959
gamma=0.529230, beta=0.597852, expected_energy=-0.00634375
gamma=0.362535, beta=0.961463, expected_energy=-0.00901344
gamma=0.584743, beta=1.294065, expected_energy=-0.01099424
gamma=0.513267, beta=1.687627, expected_energy=0.00473192
gamma=0.759402, beta=1.196626, expected_energy=-0.01688510
gamma=0.941961, beta=1.114942, expected_energy=-0.02193839
gamma=1.323981, beta=0.996363, expected_energy=-0.02863145
gamma=1.693578, beta=1.149328, expected_energy=-0.03515564
gamma=2.493535, beta=1.157622, expected_energy=-0.04601625
gamma=3.773671, beta=2.117440, expected_energy=0.12833285
gamma=2.555265, beta=0.360007, expected_energy=-0.00215332
gamma=2.583962, beta=1.547267, expected_energy=-0.00450682
gamma=2.520068, beta=0.959390, expected_energy=-0.04087605
gamma=2.592651, beta=1.170889, expected_energy=-0.04691732

In [47]:
best_gamma = [float(result.x[0])]
best_beta  = [float(result.x[1])]

print("Best gamma:", best_gamma)
print("Best beta:", best_beta)

Best gamma: [6.2150703076625105]
Best beta: [1.231185629614988]


In [48]:
main_opt = build_main_from_params(best_gamma, best_beta)
ket_opt = sim.state_vector(main_opt)
probs_opt = np.abs(np.array(ket_opt)) ** 2

top_k = 15
top_indices_opt = np.argsort(probs_opt)[::-1][:top_k]

print(f"Top {top_k} states after optimization:\n")
for rank, idx in enumerate(top_indices_opt, start=1):
    bitstring = format(idx, f"0{n_qubits}b")
    x_list = [int(b) for b in bitstring]
    energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
    print(f"{rank:2d}. bitstring={bitstring}   prob={probs_opt[idx]:.8f}   energy={energy:.8f}")

Top 15 states after optimization:

 1. bitstring=01111011   prob=0.01356722   energy=-0.11438400
 2. bitstring=01111110   prob=0.01356298   energy=-0.14306000
 3. bitstring=01101111   prob=0.01350694   energy=-0.05546400
 4. bitstring=01110111   prob=0.01337634   energy=-0.05703200
 5. bitstring=01111111   prob=0.01295645   energy=0.03233000
 6. bitstring=01111101   prob=0.01294682   energy=-0.09982200
 7. bitstring=00111111   prob=0.01273652   energy=-0.07715600
 8. bitstring=01011111   prob=0.01250680   energy=-0.01557600
 9. bitstring=01111010   prob=0.01212059   energy=-0.07643400
10. bitstring=01101011   prob=0.01098262   energy=-0.13817800
11. bitstring=01110011   prob=0.01087208   energy=-0.13972600
12. bitstring=01101110   prob=0.00998125   energy=-0.12419400
13. bitstring=01110110   prob=0.00988228   energy=-0.12575400
14. bitstring=10011111   prob=0.00966411   energy=-0.06678000
15. bitstring=01111001   prob=0.00949895   energy=-0.13986800


In [49]:
gt_index = int(ground_truth_bitstring, 2)

print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth optimized probability:", probs_opt[gt_index])
print("Ground-truth Ising energy:", ising_energy_from_x_list(
    [int(b) for b in ground_truth_bitstring], h8, J8, ising_offset
))

Ground-truth bitstring: 11011110
Ground-truth optimized probability: 0.007760842200114138
Ground-truth Ising energy: -0.14326400000000006


**After some optimization runs and getting results we are ready for a "inference cell" or a unique cell where we can just tune p (depth of the circuit), the iterations, shots, etc.** 

# **Inference cell**

In [52]:
# - -- - -- - - THESE ARE THE CONFIGURATIONS YOU CAN TUNE - - -- - - 

P = 1             # DEPTH OF THE CIRCUIT, how much we repeat Hc and Hm gates or layers. if p=2 2 layers of hc and hm
MAXITER = 5000    # Maximum iterations for the classical optimizer "COBYLA" [MAXFUN TIMES log means we got to up this variable]
RHOBEG = 0.2      # Parameter for the classical optimizer or COBYLA, is the initial step size, how aggressive we do the classical optimizer
METHOD = "COBYLA" # The optimizer, we can (and should) change for other than COBYLA after 

# multiple initializations for the classical optimizer
INITIAL_POINTS = [
    ([0.3] * P, [0.2] * P),
    ([0.5] * P, [0.5] * P),
    ([1.0] * P, [0.5] * P),
    ([2.0] * P, [1.0] * P),
    ([np.pi / 2] * P, [np.pi / 4] * P),
]

TOP_K = 15       # How many States i print at the last report
SHOTS = 2000     # How much samples i want to simulate on the QAOA [shoots]
SEED = 123       # SEED for reproducibility
VERBOSE = False  # If i want to see every evaluation of the optimizer or no

# -------- build kernel + simulator from known variables --------
kernel = build_qaoa_ising_kernel(n_qubits, h_terms, J_terms) # WE BUILD THE QAOA CIRCUIT
sim = StackMemorySimulator(min_qubits=n_qubits)   # WE CREATE THE QUANTUM SIMULATOR

# -------- helper functions local to this cell --------

# We combine gamma and beta into theta for the Optimizer
def join_theta(gamma_vals, beta_vals):
    return np.array(list(gamma_vals) + list(beta_vals), dtype=float)

# We separate theta into gamma and beta for the QAOA
def split_theta(theta, p):
    theta = np.array(theta, dtype=float)
    gamma_vals = theta[:p].tolist()
    beta_vals = theta[p:2*p].tolist()
    return gamma_vals, beta_vals

# Function to calculate expected Ising energy. the final value sum of energy x probability. IS USED FOR THE OPTIMIZER
def expected_ising_energy_for_theta(theta):
    gamma_vals, beta_vals = split_theta(theta, P)
    main = build_main_from_params(gamma_vals, beta_vals)
    ket = sim.state_vector(main)
    probs = np.abs(np.array(ket)) ** 2

    exp_energy = 0.0
    for idx in range(2 ** n_qubits):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
        exp_energy += probs[idx] * energy

    return float(exp_energy)

# We run 1 classic optimization
def optimize_once(theta0):
    eval_history = []

    def objective(theta):
        value = expected_ising_energy_for_theta(theta)
        eval_history.append((theta.copy(), value))
        if VERBOSE:
            g, b = split_theta(theta, P)
            print(f"gamma={np.round(g, 6)}, beta={np.round(b, 6)}, expected_energy={value:.8f}")
        return value

    result = minimize(
        objective,
        theta0,
        method=METHOD,
        options={"maxiter": MAXITER, "rhobeg": RHOBEG}
    )
    return result, eval_history

# Analysis wise function
def top_states_from_probs(probs, top_k=15):
    top_indices = np.argsort(probs)[::-1][:top_k]
    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
        rows.append({
            "rank": rank,
            "bitstring": bitstring,
            "prob": float(probs[idx]),
            "energy": float(energy),
        })
    return rows

def sample_counts_from_probs(probs, shots=1000, seed=123):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(probs))
    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    sampled = rng.choice(indices, size=shots, p=probs)

    counts = {}
    for idx in sampled:
        bitstring = format(idx, f"0{n_qubits}b")
        counts[bitstring] = counts.get(bitstring, 0) + 1

    return dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True))

# -------- multi-start classical optimization --------
all_runs = []
best_result = None
best_history = None
best_theta0 = None

for run_id, (gamma0, beta0) in enumerate(INITIAL_POINTS, start=1):
    theta0 = join_theta(gamma0, beta0)
    result, history = optimize_once(theta0)
    all_runs.append({
        "run_id": run_id,
        "theta0": theta0,
        "result": result,
        "history": history
    })

    if best_result is None or result.fun < best_result.fun:
        best_result = result
        best_history = history
        best_theta0 = theta0

# -------- recover best params --------
best_gamma, best_beta = split_theta(best_result.x, P)

print("===================================")
print("BEST RUN SUMMARY")
print("===================================")
print("Initial theta:", best_theta0)
print("Best theta:", best_result.x)
print("Best gamma:", best_gamma)
print("Best beta:", best_beta)
print("Best expected energy:", best_result.fun)
print("Success:", best_result.success)
print("Message:", best_result.message)

# -------- evaluate final optimized circuit --------
main_opt = build_main_from_params(best_gamma, best_beta)
ket_opt = sim.state_vector(main_opt)
probs_opt = np.abs(np.array(ket_opt)) ** 2

# -------- top states --------
rows = top_states_from_probs(probs_opt, top_k=TOP_K)

print("\n===================================")
print(f"TOP {TOP_K} STATES AFTER OPTIMIZATION")
print("===================================")
for r in rows:
    print(f"{r['rank']:2d}. bitstring={r['bitstring']}   prob={r['prob']:.8f}   energy={r['energy']:.8f}")

# -------- ground truth info --------
gt_index = int(ground_truth_bitstring, 2)
gt_energy = ising_energy_from_x_list([int(b) for b in ground_truth_bitstring], h8, J8, ising_offset)

print("\n===================================")
print("GROUND TRUTH CHECK")
print("===================================")
print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth probability:", probs_opt[gt_index])
print("Ground-truth energy:", gt_energy)

# -------- shot-based sampling --------
counts = sample_counts_from_probs(probs_opt, shots=SHOTS, seed=SEED)

print("\n===================================")
print(f"TOP 10 SAMPLED STATES ({SHOTS} shots)")
print("===================================")
for rank, (bitstring, count) in enumerate(list(counts.items())[:10], start=1):
    energy = ising_energy_from_x_list([int(b) for b in bitstring], h8, J8, ising_offset)
    print(f"{rank:2d}. bitstring={bitstring}   count={count}   freq={count/SHOTS:.6f}   energy={energy:.8f}")

BEST RUN SUMMARY
Initial theta: [2. 1.]
Best theta: [6.45751802 1.23487119]
Best gamma: [6.45751802273692]
Best beta: [1.234871190831047]
Best expected energy: -0.0637200549729433
Success: True
Message: Return from COBYLA because the trust region radius reaches its lower bound.

TOP 15 STATES AFTER OPTIMIZATION
 1. bitstring=01111011   prob=0.01389535   energy=-0.11438400
 2. bitstring=01111110   prob=0.01388738   energy=-0.14306000
 3. bitstring=01101111   prob=0.01382781   energy=-0.05546400
 4. bitstring=01110111   prob=0.01369172   energy=-0.05703200
 5. bitstring=01111111   prob=0.01328093   energy=0.03233000
 6. bitstring=01111101   prob=0.01325272   energy=-0.09982200
 7. bitstring=00111111   prob=0.01305577   energy=-0.07715600
 8. bitstring=01011111   prob=0.01288525   energy=-0.01557600
 9. bitstring=01111010   prob=0.01234288   energy=-0.07643400
10. bitstring=01101011   prob=0.01113904   energy=-0.13817800
11. bitstring=01110011   prob=0.01102479   energy=-0.13972600
12. bi